Additional Features:
 - Excess style momentum
 - Excess style volatility (against benchmark)
 - Excess style kurtosis (against benchmark)
 - Excess style skewness (against benchmark)
 - Relative momentum (factors relative to each other, trend scan - categorical features)
 - Short-technical indicators (remove)
 - Medium technical indicators (remove)
 - Omega ration (remove)


In [1]:
# run script from the data_input file
import os
os.chdir('C:/Users/p528552/OneDrive - South African Reserve Bank/Documents/1. MEng - Data Science/1. Project_2025/Data/factor_timing/data_input')

In [2]:
# Import libraries:
import numpy as np
import pandas as pd
from scipy.stats import kurtosis, skew


In [6]:
# Import data:
df_benchmark = pd.read_csv("benchmark_returns.csv", parse_dates=['Date'])
df_benchmark = df_benchmark.set_index('Date')

df_factors = pd.read_csv("factor_returns.csv", parse_dates=['Date'])
df_factors = df_factors.set_index('Date')

df_vol = pd.read_csv("local_global_vol.csv", parse_dates=['Date'])
df_vol = df_vol.set_index('Date')

df = pd.concat([df_benchmark, df_factors,df_vol], axis=1, join='inner')
df = df.dropna()

print(df_vol)
#print(df.dtypes)


                SPX     JALSH    VIX    MOVE  SAVIT40  VIX3M  VIX6M  VIX1Y
Date                                                                      
1998-10-01  1017.01   4695.37  40.95  124.22      NaN    NaN    NaN    NaN
1998-11-01  1098.67   5357.66  28.05  121.06      NaN    NaN    NaN    NaN
1998-12-01  1163.63   5202.07  26.01   87.22      NaN    NaN    NaN    NaN
1999-01-01  1229.23   5025.85  24.42  121.13      NaN    NaN    NaN    NaN
1999-02-01  1279.64   5427.55  26.25   87.16      NaN    NaN    NaN    NaN
...             ...       ...    ...     ...      ...    ...    ...    ...
2024-09-01  5648.40  83749.86  15.00  107.77    16.07  17.43  18.88  20.29
2024-10-01  5762.48  86548.42  16.73   94.61    15.25  19.22  20.62  21.85
2024-11-01  5705.45  85384.82  23.16  135.18    17.32  22.64  22.61  22.73
2024-12-01  6032.38  84510.44  13.51   95.22    16.78  16.26  18.15  19.74
2024-12-03  6047.15  85731.51  13.34   97.81    16.17  16.34  18.05  19.81

[316 rows x 8 columns]


In [4]:
df.columns

Index(['Benchmark', 'Momentum', 'Value', 'Quality', 'SPX', 'JALSH', 'VIX',
       'MOVE', 'SAVIT40', 'VIX3M', 'VIX6M', 'VIX1Y'],
      dtype='object')

In [ ]:
# Geometric and arithmetc means:

def generate_momentum_features(df, cols, windows=[1, 3, 6, 12]):

    df = df.copy()
    
    for col in cols:
        for w in windows:
            # Rolling arithmetic mean
            #df[f"{col}_rollmean_{w}m"] = df[col].rolling(window=w).mean()

            # Rolling geometric mean
            df[f"{col}_geommean_{w}m"] = (
                np.exp(np.log(df[col] + 1).rolling(window=w).mean()) - 1
            )

    return df


cols = ['Momentum','Value','Quality','SPX','JALSH']
df_mom = generate_momentum_features(df, cols)


df_mom.columns

Index(['Benchmark', 'Momentum', 'Value', 'Quality', 'SPX', 'JALSH', 'VIX',
       'MOVE', 'SAVIT40', 'VIX3M', 'VIX6M', 'VIX1Y', 'Momentum_geommean_1m',
       'Momentum_geommean_3m', 'Momentum_geommean_6m', 'Momentum_geommean_12m',
       'Momentum_geommean_24m', 'Value_geommean_1m', 'Value_geommean_3m',
       'Value_geommean_6m', 'Value_geommean_12m', 'Value_geommean_24m',
       'Quality_geommean_1m', 'Quality_geommean_3m', 'Quality_geommean_6m',
       'Quality_geommean_12m', 'Quality_geommean_24m', 'SPX_geommean_1m',
       'SPX_geommean_3m', 'SPX_geommean_6m', 'SPX_geommean_12m',
       'SPX_geommean_24m', 'JALSH_geommean_1m', 'JALSH_geommean_3m',
       'JALSH_geommean_6m', 'JALSH_geommean_12m', 'JALSH_geommean_24m'],
      dtype='object')

In [6]:
df_mom_features = df_mom[['Momentum_geommean_1m',
       'Momentum_geommean_3m', 'Momentum_geommean_6m', 'Momentum_geommean_12m',
       'Momentum_geommean_24m', 'Value_geommean_1m', 'Value_geommean_3m',
       'Value_geommean_6m', 'Value_geommean_12m', 'Value_geommean_24m',
       'Quality_geommean_1m', 'Quality_geommean_3m', 'Quality_geommean_6m',
       'Quality_geommean_12m', 'Quality_geommean_24m', 'SPX_geommean_1m',
       'SPX_geommean_3m', 'SPX_geommean_6m', 'SPX_geommean_12m',
       'SPX_geommean_24m', 'JALSH_geommean_1m', 'JALSH_geommean_3m',
       'JALSH_geommean_6m', 'JALSH_geommean_12m', 'JALSH_geommean_24m']]

df_mom_features.to_csv("momentum_features.csv", index=True)

In [7]:
# Excess momentum, volatility, kurtosis and skewness:

window = 6

for style in styles:
    # Excess Style Momentum (Cumulative excess return)
    df[f'{style}_excess_momentum'] = df[f'{style}_excess'].rolling(window).sum()
    

    # Excess Style Volatility
    df[f'{style}_excess_volatility'] = df[f'{style}_excess'].rolling(window).std()
    

    # Excess Style Kurtosis
    df[f'{style}_excess_kurtosis'] = df[f'{style}_excess'].rolling(window).apply(kurtosis, raw=True)

    # Excess Style Skewness
    df[f'{style}_excess_skewness'] = df[f'{style}_excess'].rolling(window).apply(skew, raw=True)
    
    


NameError: name 'styles' is not defined

In [ ]:
# Compare factors risk to benchmark risk:

# Rolling risk stats for benchmark
df['benchmark_vol'] = df['Benchmark'].rolling(window).std()
df['benchmark_kurtosis'] = df['Benchmark'].rolling(window).apply(kurtosis, raw=True)
df['benchmark_skewness'] = df['Benchmark'].rolling(window).apply(skew, raw=True)

for style in styles:
    df[f'{style}_relative_volatility'] = df[f'{style}'].rolling(window).std() / df['benchmark_vol']
    df[f'{style}_relative_kurtosis'] = df[f'{style}'].rolling(window).apply(kurtosis, raw=True) / df['benchmark_kurtosis']
    df[f'{style}_relative_skewness'] = df[f'{style}'].rolling(window).apply(skew, raw=True) / df['benchmark_skewness']


In [ ]:
# Technical indicators:
for style in styles:
    df[f'{style}_momentum_1M'] = df[style].shift(1)
    df[f'{style}_momentum_3M'] = df[style].rolling(3).sum()
    df[f'{style}_momentum_6M'] = df[style].rolling(6).sum()
    df[f'{style}_momentum_12M'] = df[style].rolling(12).sum()
    df[f'{style}_momentum_24M'] = df[style].rolling(24).sum()


In [ ]:
# Relative factor performance:

df = df.drop(['Benchmark','Momentum','Value','Quality','Momentum_excess','Value_excess','Quality_excess',
              'benchmark_vol','benchmark_kurtosis','benchmark_skewness'],axis=1)
print(df.columns)


Index(['Momentum_excess_momentum', 'Momentum_excess_volatility',
       'Momentum_excess_kurtosis', 'Momentum_excess_skewness',
       'Value_excess_momentum', 'Value_excess_volatility',
       'Value_excess_kurtosis', 'Value_excess_skewness',
       'Quality_excess_momentum', 'Quality_excess_volatility',
       'Quality_excess_kurtosis', 'Quality_excess_skewness',
       'Momentum_relative_volatility', 'Momentum_relative_kurtosis',
       'Momentum_relative_skewness', 'Value_relative_volatility',
       'Value_relative_kurtosis', 'Value_relative_skewness',
       'Quality_relative_volatility', 'Quality_relative_kurtosis',
       'Quality_relative_skewness', 'Momentum_momentum_1M',
       'Momentum_momentum_3M', 'Momentum_momentum_6M', 'Momentum_momentum_12M',
       'Momentum_momentum_24M', 'Value_momentum_1M', 'Value_momentum_3M',
       'Value_momentum_6M', 'Value_momentum_12M', 'Value_momentum_24M',
       'Quality_momentum_1M', 'Quality_momentum_3M', 'Quality_momentum_6M',
       

In [ ]:
# Save output into csv file:
df.to_csv('factor_momentum.csv', index=True)